# 04 — Classical Radiomics, Striatal Binding Ratios (SBR) & Ridge Baselines

**Focus:** Handcrafted Radiomic Feature Extraction, SBR Biomarkers, and Ridge Regression  
**Why it matters:** Deep learning models can overfit camera textures. A classical non-CNN radiomics arm provides essential regularizing diversity for the ensemble.

---

## Radiomic Descriptors Extracted
The radiomic feature pipeline (`featc` arm) computes 116 anatomical and intensity descriptors:
1. **Striatal Binding Ratios (SBR):** Left putamen, right putamen, left caudate, right caudate relative to non-specific background.
2. **Putamen-to-Caudate Ratios:** Quantifying anterior-to-posterior uptake gradient.
3. **Bilateral Asymmetry Indices:** Measuring left-right discrepancy.
4. **Striatal Geometric Descriptors:** Volume, sphericity, surface area, center-of-mass shift.
5. **PCA Patch Projections:** Principal component analysis of registered anatomical slabs.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.model_selection import StratifiedKFold

# Synthesize realistic feature matrix for 1,362 scans
np.random.seed(42)
n_samples = 1362
y = np.array([1]*747 + [0]*615)
np.random.shuffle(y)

# Simulate key clinical biomarkers
sbr_putamen = np.where(y == 1, np.random.normal(1.1, 0.4, n_samples), np.random.normal(2.6, 0.5, n_samples))
sbr_caudate = np.where(y == 1, np.random.normal(1.8, 0.4, n_samples), np.random.normal(2.9, 0.5, n_samples))
asymmetry_index = np.where(y == 1, np.random.exponential(18.0, n_samples) + 8.0, np.random.exponential(5.0, n_samples) + 2.0)
putamen_caudate_ratio = sbr_putamen / (sbr_caudate + 1e-4)

# Noise radiomics
noise_features = np.random.normal(0, 1, size=(n_samples, 20))

X = np.column_stack([sbr_putamen, sbr_caudate, asymmetry_index, putamen_caudate_ratio, noise_features])
feature_names = ['SBR_Putamen', 'SBR_Caudate', 'Asymmetry_Index', 'Putamen_Caudate_Ratio'] + [f'Radiomic_{i}' for i in range(20)]
df_features = pd.DataFrame(X, columns=feature_names)
df_features['is_pathologic'] = y

print(f"Radiomics Feature Matrix Shape: {X.shape}")

In [ ]:
# Train 5-Fold Stratified Ridge Logistic Regression Baseline
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(n_samples)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    
    clf = LogisticRegression(C=0.1, penalty='l2', solver='lbfgs', max_iter=500)
    clf.fit(X_train, y_train)
    oof_preds[val_idx] = clf.predict_proba(X_val)[:, 1]

oof_loss = log_loss(y, oof_preds)
oof_auc = roc_auc_score(y, oof_preds)

print(f"Ridge Radiomics OOF Log Loss: {oof_loss:.4f}")
print(f"Ridge Radiomics OOF AUROC:    {oof_auc:.4f}")

# Plot Feature Importances
weights = np.abs(clf.coef_[0][:4])
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(feature_names[:4], weights, color='#7570b3', edgecolor='black')
ax.set_xlabel('Absolute Ridge Logistic Regression Weight')
ax.set_title('Top Classical Feature Importances for DaT-SPECT Classification')
ax.invert_yaxis()
plt.tight_layout()
plt.show()